In [1]:
import json
import pickle
import numpy as np
import pandas as pd
import os
print(os.getcwd())

c:\Users\atom0\OneDrive\Documents\College_folder\4th\DACN_Do-Thanh-Thai\CL-TTE\playground


In [2]:
data_path = "../../data/hcm_data/"
train_path = os.path.join(data_path, "train.npy")
data = np.load(train_path, allow_pickle=True)
print(f"Loaded data from {train_path}, shape: {data.shape}")

Loaded data from ../../data/hcm_data/train.npy, shape: (765036, 6)


In [4]:
for d in data:
    if d[1] == []:
        print(d)
        break

In [ ]:
with open(os.path.join(data_path,"nwk_hcm/hcm_edges_poi_new_simplify.pkl"), 'rb') as f:
    edgeinfo = pickle.load(f)
with open(os.path.join(data_path,"nwk_hcm/hcm_nodes_new.pkl"), 'rb') as f:
    nodeinfo = pickle.load(f)

In [ ]:
import numpy as np
from tqdm import tqdm

In [ ]:
import numpy as np

all_edge_lengths = []
all_culm_lengths = []

for d in tqdm(data):
    edge_list = d[1]
    cum = 0
    for eid in edge_list:
        edge_len = edgeinfo[eid][1]
        all_edge_lengths.append(edge_len)
        
        cum += edge_len
        all_culm_lengths.append(cum)

# convert to numpy arrays
all_edge_lengths = np.array(all_edge_lengths)
all_culm_lengths = np.array(all_culm_lengths)

# compute mean and std
edge_mean = all_edge_lengths.mean()
edge_std = all_edge_lengths.std()

culm_mean = all_culm_lengths.mean()
culm_std = all_culm_lengths.std()

print(f"Edge length: mean={edge_mean:.3f}, std={edge_std:.3f}")
print(f"Cumulative length: mean={culm_mean:.3f}, std={culm_std:.3f}")

In [ ]:
from tqdm import tqdm
import math

# Initialize
count = 0
start_lat_mean = start_lat_M2 = 0.0
start_lon_mean = start_lon_M2 = 0.0
end_lat_mean = end_lat_M2 = 0.0
end_lon_mean = end_lon_M2 = 0.0

for d in tqdm(data):
    for eid in d[1]:
        edge = edgeinfo[eid]
        s_lat, s_lon, _ = nodeinfo[edge[2]]
        e_lat, e_lon, _ = nodeinfo[edge[3]]
        
        # Increment count
        count += 1
        
        # Start latitude
        delta = s_lat - start_lat_mean
        start_lat_mean += delta / count
        start_lat_M2 += delta * (s_lat - start_lat_mean)
        
        # Start longitude
        delta = s_lon - start_lon_mean
        start_lon_mean += delta / count
        start_lon_M2 += delta * (s_lon - start_lon_mean)
        
        # End latitude
        delta = e_lat - end_lat_mean
        end_lat_mean += delta / count
        end_lat_M2 += delta * (e_lat - end_lat_mean)
        
        # End longitude
        delta = e_lon - end_lon_mean
        end_lon_mean += delta / count
        end_lon_M2 += delta * (e_lon - end_lon_mean)

# Compute standard deviations
start_lat_std = math.sqrt(start_lat_M2 / count)
start_lon_std = math.sqrt(start_lon_M2 / count)
end_lat_std = math.sqrt(end_lat_M2 / count)
end_lon_std = math.sqrt(end_lon_M2 / count)

print(f"Start latitude: mean={start_lat_mean:.6f}, std={start_lat_std:.6f}")
print(f"Start longitude: mean={start_lon_mean:.6f}, std={start_lon_std:.6f}")
print(f"End latitude: mean={end_lat_mean:.6f}, std={end_lat_std:.6f}")
print(f"End longitude: mean={end_lon_mean:.6f}, std={end_lon_std:.6f}")

In [3]:
print(data[0])

[19497
 list([21264, 4999, 21263, 26818, 5309, 43114, 25057, 5690, 43455, 21314, 25072, 25074, 21302, 25133, 21299, 12074, 42247, 42248, 42250, 42252, 5449, 18145, 21309, 44724, 25823, 1194, 21863, 21857, 21368, 21257, 22187, 45642, 22166, 26287, 39457, 26277, 3565, 4710, 14057, 25818, 4433, 21258, 44439, 11330, 11329, 3909, 10039, 10037, 9454, 32938, 10040, 17554, 532, 46138, 40651, 45702, 12573, 45257, 12577, 45247, 12579, 45238, 45233, 12571, 12563, 12568, 40410, 12566, 17982, 45216, 45250, 2074, 9655, 45262, 9649, 10118, 10068, 40374, 26662, 4562, 26207, 13916, 13915, 17508, 40379, 27623, 22521, 22517, 23062, 16843, 23057, 23063, 23068, 23069, 23074, 22512, 22514, 22531, 16969, 22530, 22536, 22535, 28001, 27374, 12220, 4058, 16802, 11411, 11410, 31626, 12222, 11406, 16793, 30095, 32413, 29987, 30102, 44166, 32415, 20886, 23573, 871, 23570, 29255, 14204, 29243, 2543, 23095, 4690, 28536, 18905, 23108, 37830, 37834, 28533, 20911, 28465, 28476, 16674, 37776, 1504, 189, 37780, 16682, 16

In [ ]:
times = []
for d in data:
    times.append(d[-1])
print(np.mean(times), np.std(times))

In [ ]:
print(type(edgeinfo), len(edgeinfo))

In [ ]:
import ast

# 1. Extract all raw types from your edgeinfo values
raw_types = {v[0] for v in edgeinfo.values()}

# 2. Function to turn "['a', 'b']" or "a" into a flat list ['a', 'b']
def listify_string(val):
    if isinstance(val, str) and val.startswith("["):
        try:
            return ast.literal_eval(val)
        except:
            return [val]
    return [val]

# 3. Flatten everything into a single set of unique "atomic" road types
atomic_types = set()
for t in raw_types:
    atoms = listify_string(t)
    for a in atoms:
        atomic_types.add(a)

# 4. Construct the highway dict
# ID 0: Reserved for Padding
# ID 1: Reserved for 'unclassified' (The Catch-all)
highway = {"<PAD>": 0, "unclassified": 1}

# Add all other types starting from ID 2
current_id = 2
for t in sorted(list(atomic_types)):
    if t != "unclassified":
        highway[t] = current_id
        current_id += 1

print(f"Total Unique Atomic Types: {len(highway)}")
print(highway)

In [ ]:
count = {road_type : 0 for road_type in highway.keys()}

In [ ]:
print(edgeinfo[0])

In [ ]:
for d in data:
    edge_id_list = d[1]
    for edge_id in edge_id_list:
        edge = edgeinfo[edge_id]
        road_type = edge[0]
        if road_type.startswith("["):
            try:
                types = ast.literal_eval(road_type)
                for t in types:
                    if t in count:
                        count[t] += 1
            except:
                if road_type in count:
                    count[road_type] += 1
        else:
            if road_type in count:
                count[road_type] += 1
    
print(count)

In [ ]:
print(len(edgeinfo))

In [ ]:
print("First data sample:")
print(data[0])

In [ ]:
def parse_highway_tags(raw_val, max_tags=2):
    """Converts OSM strings/lists to a fixed-size list of IDs."""
    UNCLASSIFIED_ID = highway.get('unclassified', 1)
    
    # 1. Handle string/list input
    if isinstance(raw_val, str) and raw_val.startswith("["):
        try: tags = ast.literal_eval(raw_val)
        except: tags = [raw_val]
    elif isinstance(raw_val, list):
        tags = raw_val
    else:
        tags = [raw_val]

    # 2. Map to IDs with fallback
    ids = [highway.get(t, UNCLASSIFIED_ID) for t in tags]
    
    # 3. Pad with 0 (Reserved for 'No Tag')
    while len(ids) < max_tags:
        ids.append(0)
    return ids[:max_tags]

In [ ]:
id_ = parse_highway_tags(edgeinfo[0][0])
print("Raw highway tag for edge 0:", edgeinfo[0][0])
print(f"Parsed IDs for edge 0: {id_}")

In [ ]:
print("STD: ", np.std(data[:, -1]))

In [ ]:
os.listdir('../')